# Phase 1 — build the dataset

Ingest source datasets → manifest → grouped split → verify → stats → YOLO export.

The manifest is the source of truth; the YOLO tree is regenerated from it. Everything here is a thin call into `alpr.data` — the logic is tested in CI, not in this notebook.

**Detection is trained region-agnostically.** A plate detector mostly learns "small bright rectangle on a vehicle" and transfers across countries; the India/Germany split lives in Phase 5's grammar validators, which need no data at all. That is what lets us use whatever is openly licensed.

In [ ]:
REPO = "https://github.com/fayazhussain2821/Automatic-License-Plate-Recognition.git"
BRANCH = "main"

import os

if os.path.isdir("/content/ALPR"):
    !cd /content/ALPR && git fetch --quiet origin && git checkout --quiet $BRANCH && git pull --quiet
else:
    !git clone --quiet --branch $BRANCH $REPO /content/ALPR

%cd /content/ALPR
!pip install --quiet -e .
!git log --oneline -1

# An editable install registers itself through a .pth file in site-packages,
# and .pth files are only processed at interpreter startup. This kernel was
# already running when pip ran, so it cannot see the package — `import alpr`
# fails with ModuleNotFoundError even though the install above succeeded.
# Adding the source directory explicitly avoids needing a runtime restart.
import sys

SRC = "/content/ALPR/src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import alpr

print(f"alpr {alpr.__version__} importable")

## 1. Acquire sources

Two Roboflow Universe datasets, both **CC BY 4.0** — redistributable with attribution, which matters because Phase 1 publishes a *derived* dataset.

| Source | Images | Classes | Licence |
|---|---|---|---|
| [`e-hh49k/european-license-plates-tjviy`](https://universe.roboflow.com/e-hh49k/european-license-plates-tjviy) | 1,455 | 1 | CC BY 4.0 |
| [`nivu/indian-license-plate-knte7`](https://universe.roboflow.com/nivu/indian-license-plate-knte7) | 1,650 | 1 | CC BY 4.0 |

**3,105 images**, comfortably past the 1,000-plate exit criterion.

The European set is tagged `Region.EUROPE`, **not** `GERMANY` — it holds French, Spanish, Italian and other EU plates, and claiming otherwise would be a false statement about the data. That costs nothing here: a plate detector transfers across countries, and the German-specific work is Phase 5's grammar, which needs no training data.

Keys come from Colab's **🔑 Secrets** panel (left sidebar), named `ROBOFLOW_API_KEY` and `KAGGLE_API_TOKEN`, with *Notebook access* switched on. Never paste a key into a cell — cells get committed, and this repo is public.

In [ ]:
import os
from pathlib import Path

from alpr.env import ROBOFLOW_KEY_VAR, get_credential

RAW = Path("data/raw")
RAW.mkdir(parents=True, exist_ok=True)

# Reads the key from Colab's 🔑 Secrets panel (or the environment) and raises
# with instructions if it is not configured. Nothing is printed: echoing a key
# writes it into the notebook's saved output.
#
# NEVER paste a key into this cell. Cells are committed, this repo is public,
# and GitHub keeps pushed objects reachable long after a commit is amended
# away — a leaked key has to be rotated, not deleted.
os.environ[ROBOFLOW_KEY_VAR] = get_credential(ROBOFLOW_KEY_VAR)

!pip install --quiet roboflow

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=os.environ[ROBOFLOW_KEY_VAR])

# (workspace, project, version, local directory)
DOWNLOADS = [
    ("e-hh49k", "european-license-plates-tjviy", 1, "roboflow-eu"),
    ("nivu", "indian-license-plate-knte7", 1, "roboflow-in"),
]

for workspace, project, version, name in DOWNLOADS:
    target = RAW / name
    if target.exists():
        print(f"{name}: already downloaded, skipping")
        continue
    print(f"{name}: downloading {workspace}/{project} v{version} ...")
    rf.workspace(workspace).project(project).version(version).download(
        "yolov8", location=str(target)
    )

!find {RAW} -maxdepth 3 -type d | sort

## 2. Ingest into one manifest

`from_roboflow_export` pools each export's `train/valid/test` directories into a single set of records.

**Roboflow's own split is discarded deliberately.** It is random per image, so near-duplicate shots of one scene can land on both sides of it, and it knows nothing about our region balance. We re-split in step 3 with our own grouped, stratified, deterministic logic — trusting the upstream split would quietly reintroduce exactly the leakage Phase 1 exists to prevent.

In [ ]:
from alpr.data import Region, from_roboflow_export

# (export dir, source tag, region, id prefix)
#
# The `source` tag is the licence audit trail — both of these are CC BY 4.0
# and so redistributable, which step 6 relies on.
#
# `id_prefix` keeps the two sources from colliding: both exports contain
# generically-named files, and without a prefix one would overwrite the other.
SOURCES = [
    (RAW / "roboflow-eu", "roboflow-eu-plates", Region.EUROPE, "eu-"),
    (RAW / "roboflow-in", "roboflow-in-plates", Region.INDIA, "in-"),
]

records = []
for location, source, region, prefix in SOURCES:
    report = from_roboflow_export(
        location,
        source=source,
        region=region,
        id_prefix=prefix,
        path_root=RAW,  # paths relative to the shared root so sources compose
        strict=False,  # record bad labels rather than aborting a long ingest
    )
    print(f"--- {source} ({region.value}) ---")
    print(report.summary())
    print()
    records.extend(report.records)

print(f"total: {len(records)} images, {sum(len(r.boxes) for r in records)} plates")


## 2b. Merge near-duplicates, then write the manifest

Filename grouping only catches what a filename admits to. Roboflow exports augmented copies of one photograph under unrelated names, and public datasets get assembled from overlapping sources — neither is visible to a naming rule, and either puts an image the model trained on into the test split.

`group_duplicates` hashes every image perceptually (dHash) and forces near-duplicates into one split group, merged with the filename groups rather than replacing them. This is the same call `alpr.build.build_dataset` makes, so this notebook and `ensure_dataset()` produce identical splits.

**The regrouping happens before `write_manifest`, not after.** The manifest is the source of truth: the groups the split is computed from have to be *in* it, or a later `read_manifest()` + `split_records()` — which notebook 03 does to pick the test images — would disagree with the tree that was exported.


In [ ]:
from alpr.build import group_duplicates
from alpr.data import write_manifest

# The report describes the split filename grouping alone would have produced,
# so the contamination it names is what this step goes on to prevent.
records, duplicates = group_duplicates(records, RAW, seed=0)
print(duplicates.report())

merged = sum(1 for r in records if r.group and r.group.startswith("dup:"))
print(f"\n{merged} image(s) merged into shared duplicate groups")

write_manifest(records, "data/manifest.jsonl")


## 3. Split, and prove it does not leak

Grouped (frames from one clip stay together), stratified by region, and deterministic from the seed. `verify_split` raises rather than warns — a bad split is worth stopping for, because everything downstream is measured against it.

In [ ]:
from alpr.data import read_manifest, split_records, verify_split

SEED = 0

records = read_manifest("data/manifest.jsonl")
assignment = split_records(records, seed=SEED)
verify_split(records, assignment)

print("split verified — no leakage, full coverage")
for split, count in sorted(assignment.counts(records).items()):
    print(f"  {split.value:<6} {count}")

## 4. Stats and the Phase 1 exit criteria

In [ ]:
from alpr.data import check_exit_criteria, compute_stats

stats = compute_stats(records, assignment)
print(stats.report())

failures = check_exit_criteria(stats)
print()
if failures:
    print("Phase 1 NOT met:")
    for f in failures:
        print(f"  - {f}")
else:
    print("Phase 1 exit criteria met.")

The `tiny (<32px wide)` figure is the one to watch. Those plates are unreadable in principle — the detector may find them but Phase 4's OCR has no strokes to work with. A high share caps end-to-end accuracy no matter how good the detector gets.

## 5. Export the YOLO tree for Phase 2

In [ ]:
from alpr.data import export_yolo

result = export_yolo(
    records,
    assignment,
    image_root="data/raw",
    out_root="data/yolo",
    symlink=True,  # copying tens of thousands of JPEGs wastes the session
)
print(result.summary())
print()
print(result.data_yaml.read_text())

## 6. Publish

Push the manifest and the split to the Hub so Phase 2 trains against a fixed, named dataset version rather than whatever happened to be in this session.

**Exclude any source whose licence forbids redistribution** (the CC BY-NC-ND one) before uploading images.

In [ ]:
# Both sources are CC BY 4.0, so both may be redistributed with attribution.
# Any future CC BY-NC-ND source must be excluded here: NoDerivatives forbids
# publishing a re-split, which is exactly what this pipeline produces.
REDISTRIBUTABLE = {"roboflow-eu-plates", "roboflow-in-plates"}

ATTRIBUTION = """\
Derived from two Roboflow Universe datasets, both CC BY 4.0:
- European License Plates — https://universe.roboflow.com/e-hh49k/european-license-plates-tjviy
- Indian License Plate (NIVU) — https://universe.roboflow.com/nivu/indian-license-plate-knte7
"""

publishable = [r for r in records if r.source in REDISTRIBUTABLE]
print(f"{len(publishable)} of {len(records)} records are redistributable")
print()
print(ATTRIBUTION)

# from huggingface_hub import HfApi
# write_manifest(publishable, "data/manifest_public.jsonl")
# Path("data/ATTRIBUTION.txt").write_text(ATTRIBUTION)
# api = HfApi()
# api.upload_file(
#     path_or_fileobj="data/manifest_public.jsonl",
#     path_in_repo="manifest.jsonl",
#     repo_id="Babblu2821/alpr-plates",
#     repo_type="dataset",
# )
# api.upload_file(
#     path_or_fileobj="data/ATTRIBUTION.txt",
#     path_in_repo="ATTRIBUTION.txt",
#     repo_id="Babblu2821/alpr-plates",
#     repo_type="dataset",
# )